# DuckPD Transformers Series Embeddings

This notebook exercises DuckPD's first-party Transformers series backend with a pinned, public, tiny PatchTSMixer backbone. It builds seven-channel rolling windows, prepares an attested immutable model cache, embeds a corpus in bounded Arrow batches, searches with raw and pre-encoded queries, persists representation identity, and introduces temporal and static input contracts.

### What you will learn
- How a pinned bare Transformers backbone becomes an immutable `EmbeddingModelSpec`.
- How channel roles, model-side normalization, pooling, and provider ABI identify the vector space.
- How `TransformersSeriesEmbeddingProvider` downloads once, verifies its cache, and runs on CPU or GPU.
- How corpus and query windows cross the same canonical Arrow boundary.
- How typed raw queries, persisted vectors, and representation fingerprints prevent cross-space search.
- How temporal, cadence, and static declarations extend encoder-decoder inputs.

> The checkpoint is randomly initialized and exists for integration testing. Its nearest-neighbor geometry is not evidence of retrieval quality. Qualify a task-relevant pinned checkpoint against a native baseline before production use.


## 1. Configure a pinned Transformers series model

The standard development environment includes the optional PyTorch, Transformers, and Hugging Face Hub dependencies used here. The checkpoint is about 1 MB and runs on CPU; a CUDA or ROCm-enabled kernel is selected automatically when available. Preparation never silently changes the requested device.

PatchTSMixer uses all seven channels symmetrically inside its backbone. DuckPD still records one semantic target and six past covariates so the same representation contract can be compared with role-aware encoder-decoder models.


In [1]:
import importlib
import json
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pandas

import duckpd as pd

try:
    torch = importlib.import_module("torch")
    importlib.import_module("transformers")
    importlib.import_module("huggingface_hub")
except (ImportError, OSError) as error:
    raise RuntimeError(
        "Install DuckPD's Transformers series runtime before running this notebook."
    ) from error

MODEL_ID = "hf-internal-testing/tiny-random-PatchTSMixerModel"
MODEL_REVISION = "39dae7ec6e7807d6ae2e5bbf160450ea964471fc"
WINDOW = 512
DIMENSION = 48
CHANNELS = (
    "target",
    "trend",
    "daily_sin",
    "daily_cos",
    "weekly_sin",
    "weekly_cos",
    "volatility",
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL = pd.embedding_model(
    MODEL_ID,
    revision=MODEL_REVISION,
    dimension=DIMENSION,
    backend="transformers",
    normalize=True,
    pooling="mean-channels-patches-v1",
    input=pd.series_embedding_input(
        length=WINDOW,
        channels=CHANNELS,
        roles=("target",) + ("past_covariate",) * 6,
        normalization="patchtsmixer-config-scaling-v1",
        provider_abi="transformers-series-v1",
    ),
)
REPRESENTATION = pd.series_representation(
    window=WINDOW,
    channels=CHANNELS,
    sampling="observations",
    data_contract="tutorial/synthetic-seven-channel/v1",
    normalization="none",
    unit_norm=True,
    zero_scale="error",
    encoder=MODEL,
)

print(f"DuckPD version: {pd.__version__}")
print(f"Device: {DEVICE}")
print(f"Model fingerprint: {MODEL.fingerprint}")
print(f"Representation: FLOAT[{REPRESENTATION.dimension}]")
print(f"Representation fingerprint: {REPRESENTATION.fingerprint}")

DuckPD version: 0.1.4
Device: cuda
Model fingerprint: f6e4acda8ca1093887a32ed7f1b8036b2e069c99f190b5193f24d15bce08f37c
Representation: FLOAT[48]
Representation fingerprint: 7b5ba08f5b1eb2a40faeedac338831db004dac1eb0e92d31ab0fe4ffa62f2c28


## 2. Prepare the attested bare backbone

Constructing the provider is side-effect free. `prepare_embedding_model()` downloads only allowlisted model artifacts on the first run, verifies the pinned revision and exact bare architecture, promotes an immutable cache, and returns the observed runtime identity. Later runs verify and reuse that cache.


In [2]:
DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path("..")
CACHE_DIR = DEMO_DIR / ".cache" / "transformers-series"
ARTIFACT_DIR = DEMO_DIR / ".artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
EMBEDDED_DATA = ARTIFACT_DIR / "transformers-series-embedded.parquet"
EMBEDDED_SIDECAR = Path(f"{EMBEDDED_DATA}.duckpd-embeddings.json")

session = pd.connect(memory_limit="1GB", threads=4)
provider = pd.TransformersSeriesEmbeddingProvider(
    MODEL,
    device=DEVICE,
    cache_dir=CACHE_DIR,
)
session.register_embedding_provider(MODEL, provider)

started = perf_counter()
prepared = session.prepare_embedding_model(MODEL)
preparation_seconds = perf_counter() - started

assert prepared.model_fingerprint == MODEL.fingerprint
assert prepared.backend == "transformers"
print(f"Runtime: {prepared.execution_providers}")
print(f"Verified cache: {prepared.cache_path}")
print(f"Artifact digest: {prepared.artifact_digest}")
print(f"Preparation time: {preparation_seconds:.3f}s")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/199k [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Runtime: ('PyTorchROCm',)
Verified cache: ../.cache/transformers-series/f6e4acda8ca1093887a32ed7f1b8036b2e069c99f190b5193f24d15bce08f37c
Artifact digest: c293e10ed0ec719a4992f42dadf514f482241eef4507ca73edb01282a3f0f261
Preparation time: 2.417s


## 3. Build deterministic seven-channel windows

Two synthetic assets receive 520 hourly observations. Grouped rolling creates 512-observation arrays in chronological order, leaving 511 null warm-up rows per asset. The resulting 18 complete rows are deliberately small enough for an interactive CPU run.


In [3]:
hours = 520
records = []
for asset_number, asset in enumerate(("ALPHA", "BETA")):
    index = np.arange(hours, dtype=np.float64)
    phase = index + 5.0 * asset_number
    target = 0.002 * np.sin(phase / 11.0) + 0.0004 * np.cos(phase / 3.0)
    trend = (index / hours - 0.5) * (asset_number + 1)
    daily_sin = np.sin(2.0 * np.pi * phase / 24.0)
    daily_cos = np.cos(2.0 * np.pi * phase / 24.0)
    weekly_sin = np.sin(2.0 * np.pi * phase / 168.0)
    weekly_cos = np.cos(2.0 * np.pi * phase / 168.0)
    volatility = 0.001 + 0.0005 * np.abs(np.sin(phase / 17.0))
    timestamps = pandas.date_range("2026-01-01", periods=hours, freq="h", tz="UTC")
    records.extend(
        zip(
            timestamps,
            [asset] * hours,
            target,
            trend,
            daily_sin,
            daily_cos,
            weekly_sin,
            weekly_cos,
            volatility,
            strict=True,
        )
    )

source = pandas.DataFrame(records, columns=("timestamp", "asset", *CHANNELS))
frame = session.from_pandas(source, order_by=["asset", "timestamp"])
windows = frame.assign(
    target_window=lambda value: value.groupby("asset")["target"].rolling(WINDOW).to_array(),
    trend_window=lambda value: value.groupby("asset")["trend"].rolling(WINDOW).to_array(),
    daily_sin_window=lambda value: value.groupby("asset")["daily_sin"].rolling(WINDOW).to_array(),
    daily_cos_window=lambda value: value.groupby("asset")["daily_cos"].rolling(WINDOW).to_array(),
    weekly_sin_window=lambda value: value.groupby("asset")["weekly_sin"].rolling(WINDOW).to_array(),
    weekly_cos_window=lambda value: value.groupby("asset")["weekly_cos"].rolling(WINDOW).to_array(),
    volatility_window=lambda value: value.groupby("asset")["volatility"].rolling(WINDOW).to_array(),
)

print(f"Input rows: {len(source):,}")
print(f"Complete windows: {2 * (hours - WINDOW + 1)}")
print(f"Executions after planning: {session.execution_count}")

Input rows: 1,040
Complete windows: 18
Executions after planning: 1


## 4. Embed the corpus through the canonical Arrow boundary

`embed_series()` maps semantic channels to physical window columns. The plan remains lazy; the provider is called only by `collect()` or another execution boundary. Complete rows are packed as non-null fixed-size `float32` lists, inferred in bounded batches, mean-pooled across channels and patches, and unit-normalized according to the representation.


In [4]:
embedded = windows.embed_series(
    columns={channel: f"{channel}_window" for channel in CHANNELS},
    into="transformer_embedding",
    representation=REPRESENTATION,
    batch_size=8,
)
explanation = json.loads(embedded.explain(mode="json"))
operation = explanation["execution_boundaries"]["embedding_operations"][0]
assert operation["backend"] == "transformers"
assert operation["batch_size"] == 8
print(operation)

candidates = embedded[embedded["transformer_embedding"].notna()]
preview = candidates[["asset", "timestamp", "transformer_embedding"]].collect()
assert len(preview) == 18
assert all(len(vector) == DIMENSION for vector in preview["transformer_embedding"])
preview.head(3)

{'operation': 'embed_series', 'backend': 'transformers', 'model_fingerprint': 'f6e4acda8ca1093887a32ed7f1b8036b2e069c99f190b5193f24d15bce08f37c', 'model_prepared': True, 'representation_fingerprint': '7b5ba08f5b1eb2a40faeedac338831db004dac1eb0e92d31ab0fe4ffa62f2c28', 'dimension': 48, 'normalization': 'none', 'internal_normalization': 'patchtsmixer-config-scaling-v1', 'pooling': 'mean-channels-patches-v1', 'input_roles': ['target', 'past_covariate', 'past_covariate', 'past_covariate', 'past_covariate', 'past_covariate', 'past_covariate'], 'unit_norm': True, 'batch_size': 8, 'null_policy': 'propagate', 'channels': ['target', 'trend', 'daily_sin', 'daily_cos', 'weekly_sin', 'weekly_cos', 'volatility'], 'temporal_input': False, 'series_start_input': False, 'static_inputs': [], 'boundary': 'arrow_series_embedding_provider', 'persistence': 'lazy'}


,asset,timestamp,transformer_embedding
0,ALPHA,2026-01-22 07:00:00+00:00,"[-0.06795055, 0.15705289, -0.051371425, 0.0199..."
1,ALPHA,2026-01-22 08:00:00+00:00,"[-0.06798615, 0.15493035, -0.052355144, 0.0213..."
2,ALPHA,2026-01-22 09:00:00+00:00,"[-0.0680949, 0.15225175, -0.052755214, 0.02282..."


## 5. Search with rich and pre-encoded queries

`series_query()` snapshots caller-owned values into an immutable typed query. `search_series()` applies the same model contract to that raw query. `Session.embed_series_query()` performs that query inference once and returns a fingerprinted vector suitable for reuse. Both paths must rank the source window first at distance zero.


In [5]:
query_row = candidates[[*(f"{channel}_window" for channel in CHANNELS)]].head(1)
query_values = {
    channel: tuple(float(item) for item in query_row.iloc[0][f"{channel}_window"])
    for channel in CHANNELS
}
raw_query = pd.series_query(query_values)
typed_query = session.embed_series_query(raw_query, representation=REPRESENTATION)

raw_matches = candidates.vector.search_series(
    raw_query,
    column="transformer_embedding",
    representation=REPRESENTATION,
    metric="cosine",
    k=5,
    tie_breaker="timestamp",
)[["asset", "timestamp", "_distance"]].collect()
typed_matches = candidates.vector.search(
    typed_query,
    column="transformer_embedding",
    metric="cosine",
    k=5,
    tie_breaker="timestamp",
)[["asset", "timestamp", "_distance"]].collect()

assert raw_matches.equals(typed_matches)
assert abs(float(raw_matches.iloc[0]["_distance"])) < 1e-5
raw_matches

,asset,timestamp,_distance
0,ALPHA,2026-01-22 07:00:00+00:00,0.000000
1,ALPHA,2026-01-22 08:00:00+00:00,0.000094
2,ALPHA,2026-01-22 09:00:00+00:00,0.000349
3,BETA,2026-01-22 09:00:00+00:00,0.000436
4,ALPHA,2026-01-22 14:00:00+00:00,0.000436


## 6. Declare temporal and static encoder inputs

Role-aware Time Series Transformer, Informer, and Autoformer backbones can require generated calendar features and per-series static values. The immutable contract below declares an hourly New York cadence, DST-aware endpoint semantics, GluonTS-compatible calendar features, logarithmic age, one real static feature, and one bounded categorical feature. A matching pinned bare checkpoint must be qualified before preparation; this cell only exercises the public contract and strict round trip.


In [6]:
rich_input = pd.series_embedding_input(
    length=24,
    channels=("target", "promotion"),
    roles=("target", "past_covariate"),
    normalization="hf-time-series-scaler-v1",
    provider_abi="transformers-series-v1",
    temporal=pd.series_temporal_input(
        cadence=pd.series_cadence("hour"),
        timezone="America/New_York",
        anchor="last",
        recipe="gluonts-calendar-v1",
        features=("hour_of_day", "day_of_week", "age_log10"),
    ),
    static=(
        pd.series_static_input("scale", kind="real"),
        pd.series_static_input("series_id", kind="categorical", cardinality=3),
    ),
)
rich_query = pd.series_query(
    {"target": tuple(range(24)), "promotion": (0.0,) * 24},
    time=datetime(2026, 3, 9, 12, tzinfo=UTC),
    series_start=datetime(2026, 3, 1, 12, tzinfo=UTC),
    static={"scale": 1.25, "series_id": 2},
)
round_trip = pd.SeriesEmbeddingInputSpec.from_dict(rich_input.to_dict())
assert round_trip == rich_input
assert rich_query.static == (("scale", 1.25), ("series_id", 2))
rich_input.to_dict()

{'kind': 'series',
 'length': 24,
 'channels': ['target', 'promotion'],
 'roles': ['target', 'past_covariate'],
 'normalization': 'hf-time-series-scaler-v1',
 'schema_version': 2,
 'provider_abi': 'transformers-series-v1',
 'temporal': {'cadence': {'unit': 'hour', 'multiple': 1, 'mode': 'elapsed'},
  'timezone': 'America/New_York',
  'anchor': 'last',
  'recipe': 'gluonts-calendar-v1',
  'features': ['hour_of_day', 'day_of_week', 'age_log10']},
 'static': [{'name': 'scale',
   'kind': 'real',
   'cardinality': None,
   'normalization': 'none'},
  {'name': 'series_id',
   'kind': 'categorical',
   'cardinality': 3,
   'normalization': 'none'}]}

## 7. Persist and restore representation identity

DuckPD writes a managed sidecar next to Parquet output. Reloading restores the full model, provider ABI, channel, pooling, and representation identity. A previously encoded typed query can therefore search the restored vectors without repeating configuration.


In [7]:
embedded.write_parquet(EMBEDDED_DATA, overwrite=True)
restored = session.read_parquet(EMBEDDED_DATA)
restored_matches = (
    restored[restored["transformer_embedding"].notna()]
    .vector.search(
        typed_query,
        column="transformer_embedding",
        metric="cosine",
        k=3,
        tie_breaker="timestamp",
    )[["asset", "timestamp", "_distance"]]
    .collect()
)
assert abs(float(restored_matches.iloc[0]["_distance"])) < 1e-5
print("Persisted representation search succeeded.")

session.close()
EMBEDDED_DATA.unlink(missing_ok=True)
EMBEDDED_SIDECAR.unlink(missing_ok=True)
print("Session closed and generated data artifacts removed.")

Persisted representation search succeeded.
Session closed and generated data artifacts removed.
